# Contextual Bayesian Optimization：初めてのAsk-and-Tell

このNotebookでは、**操作できる設定値**と**観測するContext**を区別し、候補提案→実験→結果登録→方策評価を順番に実行します。

関連：[理論](../../../docs/theory/29_contextual_bayesian_optimization.md) / [実装ガイド](../../../docs/guide/29_contextual_bayesian_optimization.md)

## 1. 問題設定

- `temperature`：自分で設定できる温度（意思決定）
- `ambient`：実験時に観測する外気条件（Context）
- 品質：`1 - (temperature - ambient)**2`（説明用の模擬目的関数）

目的は単一の温度ではなく、外気条件に応じて温度を変える方策を得ることです。

### 理論：なぜContextもGPの入力に含めるのか

観測は $y_i=f(x_i,c_i)+\\varepsilon_i$ です。GPの入力は $(x_i,c_i)$、予測するのは新しい組 $(x,c)$ における目的値です。

- **学習時**：過去のすべての $(x_i,c_i,y_i)$ を使用します。
- **提案時**：今観測した $c$ を固定し、$x$ のみを探索します。
- **実験時**：実験装置に渡すのは $x$、Contextは観測値として記録します。

このNotebookでは観測雑音なしの模擬目的関数を使います。GPのノイズ仮定とは区別してください。

### 期待される挙動

品質は $1-(x-c)^2$ なので、理論的には $x=c$ が最大値1を与えます。ただし学習したBOが必ずこの値を提案するわけではありません。初期探索、GPの近似、獲得関数最適化の影響があります。

In [ ]:
from robotorchan.study import create_study

study = create_study(
    search_space={"temperature": (0.0, 1.0), "ambient": (0.0, 1.0)},
    context_names=("ambient",),
    n_initial_trials=3,
    seed=7,
    optimizer_kwargs={"num_restarts": 2, "raw_samples": 16},
)

## 2. Contextを観測して候補を提案

`ask(context=...)`は外気条件を固定して温度を提案します。初期候補を蓄積した後はGPを学習し、獲得関数を最適化します。`tell()`には**実測値**を渡します。

In [ ]:
contexts = (0.2, 0.8, 0.4, 0.6, 0.3)
for ambient in contexts:
    trial = study.ask(context={"ambient": ambient})
    temperature = trial.params["temperature"]
    quality = 1.0 - (temperature - ambient) ** 2
    study.tell(trial, quality)
    print(f"trial={trial.number}, ambient={ambient:.2f}, temperature={temperature:.3f}, quality={quality:.3f}")

## 3. 次のContextに対する候補

学習済みの履歴を使って、新しく観測したContextに対する設定値を提案します。これは**次の実験候補**であり、最適方策の証明ではありません。

In [ ]:
next_trial = study.ask(context={"ambient": 0.7})
print(next_trial.params)
# 実験した場合のみ: study.tell(next_trial, measured_quality)

## 4. 既知の方策の期待値を評価

ここではGPの予測ではなく、あらかじめ与えた2つの方策の目的値を比較します。

In [ ]:
import torch
from robotorchan.optim.contextual_policy import expected_policy_value

contexts_eval = torch.tensor([0.2, 0.5, 0.8], dtype=torch.double)
fixed = torch.full_like(contexts_eval, 0.5)
ideal = contexts_eval
outcomes = torch.stack((1 - (fixed - contexts_eval).square(), 1 - (ideal - contexts_eval).square()))
weights = torch.ones_like(contexts_eval)
print("固定設定と理論最適方策の期待値:", expected_policy_value(outcomes, weights))

## 5. 解釈と次のステップ

理論最適方策 `temperature=ambient` は目的関数を既知として導いたものです。**学習したStudyの未知Contextにおける性能ではありません**。学習済み手法の比較は `benchmarks/contextual_study.py` を参照してください。異なるContextを同時に渡す `ask_contexts` は各Context独立のq=1提案であり、joint q-batchではありません。

## 6. 観測履歴を可視化

以下では提案された設定値とContextの関係を描画します。破線 $x=c$ は**既知の模擬関数から導いた理論最適線**であり、GPの予測結果ではありません。

In [ ]:
import matplotlib.pyplot as plt

completed = [t for t in study.trials if t.value is not None]
observed_c = [t.params["ambient"] for t in completed]
observed_x = [t.params["temperature"] for t in completed]
fig, ax = plt.subplots()
ax.scatter(observed_c, observed_x, label="BO proposals")
ax.plot([0, 1], [0, 1], "--", label="analytical optimum x=c")
ax.set(xlabel="observed context (ambient)", ylabel="proposed decision (temperature)")
ax.legend()
plt.show()

## 7. Monte Carloによる期待方策価値の確認

有限Context分布の重み付き平均と、Contextをサンプリングした近似を比較します。これは**既知の方策に対する評価**であり、学習したStudyの汎化性能ではありません。

In [ ]:
from robotorchan.optim.contextual_policy import monte_carlo_policy_value

exact_values = expected_policy_value(outcomes, weights)
sampled_values = monte_carlo_policy_value(
    outcomes, weights, n_samples=10000,
    generator=torch.Generator().manual_seed(123),
)
print("exact:", exact_values)
print("MC:", sampled_values)
print("absolute difference:", (exact_values - sampled_values).abs())

## 8. 異なるContextの候補をまとめて提案する場合

`ask_contexts` は複数Contextそれぞれに対して **独立にq=1** の探索を行います。Context間の候補をまとめたjoint acquisitionではありません。候補間制約も使用できません。次の例では結果をまだ観測していないため、`tell` は呼びません。

In [ ]:
different_trials = study.ask_contexts([
    {"ambient": 0.15},
    {"ambient": 0.85},
])
for trial in different_trials:
    print(trial.number, trial.params)

## 9. 実験結果を読む際の注意

- 提案が $x=c$ に近いことと、未知Contextに対する平均性能が高いことは同義ではありません。
- 実際の比較では、学習に使わないContextを別途用意し、同じ予算・複数シードで評価します。
- 既知の模擬関数で計算した理論最適値は、実機の品質改善を保証しません。

### 参考文献

- Krause & Ong (2011), *Contextual Gaussian Process Bandit Optimization*, NeurIPS.
- Rasmussen & Williams (2006), *Gaussian Processes for Machine Learning*, Chapter 2.
- Jones, Schonlau & Welch (1998), *Efficient Global Optimization of Expensive Black-Box Functions*.
- Ament et al. (2023), *Unexpected Improvements to Expected Improvement for Bayesian Optimization*.

詳しい定式化と参考文献は[理論章](../../../docs/theory/29_contextual_bayesian_optimization.md)を参照してください。